In [16]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from sklearn.metrics import f1_score, classification_report, hamming_loss, accuracy_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.multioutput import ClassifierChain
import mlflow
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 0. SETUP MLflow
# ==========================================
root_path = Path.cwd().parent
mlflow.set_tracking_uri((root_path / "mlruns").as_uri())
mlflow.set_experiment("08b_Threshold_Optimization")

# ==========================================
# 1. LOAD DATA
# ==========================================
print("⏳ 1. Memuat Data...")

train_df = pd.read_csv(root_path / "Data/processed/train_selected_features.csv")
test_df  = pd.read_csv(root_path / "Data/split/test_data.csv")

selected_features = [
    'TIPI1', 'TIPI2', 'TIPI3', 'TIPI4', 'TIPI5', 'TIPI6', 'TIPI7', 'TIPI8', 'TIPI9', 'TIPI10',
    'VCL1', 'VCL4', 'VCL5', 'VCL6', 'VCL7', 'VCL8', 'VCL9', 'VCL10', 'VCL12', 'VCL13',
    'VCL14', 'VCL15', 'gender', 'engnat', 'age', 'religion', 'orientation',
    'race', 'voted', 'married', 'familysize'
]

target_cols = ['risk_depression', 'risk_anxiety', 'risk_stress']

X_full_train = train_df[selected_features]
Y_full_train = train_df[target_cols].astype(int)  # ← Fix: pastikan integer

X_test = test_df[selected_features]
Y_test = test_df[target_cols].astype(int)          # ← Fix: pastikan integer

print(f"✓ Full Train : {X_full_train.shape}")
print(f"✓ Test       : {X_test.shape}")

# ==========================================
# 2. SPLIT TRAIN → TRAIN_SUB + VALIDATION
# ==========================================
print("\n🔀 2. Split Train → Train_sub & Validation (80-20)")

X_train, X_val, Y_train, Y_val = train_test_split(
    X_full_train, Y_full_train,
    test_size=0.2,
    random_state=42
)

print(f"✓ Train_sub  : {X_train.shape}")
print(f"✓ Validation : {X_val.shape}")
print(f"✓ Test       : {X_test.shape}")

# ==========================================
# 3. LOAD BEST PARAMS DARI PICKLE & REBUILD MODEL
# ==========================================
print("\n📦 3. Load Model & Rebuild dari Scratch...")

model_path = root_path / "models/multilabel_xgboost_classifier_chain.pkl"
with open(model_path, "rb") as f:
    loaded_model = pickle.load(f)

# Ambil best params dari model pickle
best_params = {}
if hasattr(loaded_model, 'estimators_') and len(loaded_model.estimators_) > 0:
    sample_est  = loaded_model.estimators_[0]
    best_params = {
        'n_estimators'    : getattr(sample_est, 'n_estimators',    300),
        'max_depth'       : getattr(sample_est, 'max_depth',       3),
        'learning_rate'   : getattr(sample_est, 'learning_rate',   0.05),
        'subsample'       : getattr(sample_est, 'subsample',       0.8),
        'colsample_bytree': getattr(sample_est, 'colsample_bytree',0.7),
        'gamma'           : getattr(sample_est, 'gamma',           0.2),
        'min_child_weight': getattr(sample_est, 'min_child_weight',1),
    }

print(f"✓ Best params dari pickle:")
for k, v in best_params.items():
    print(f"   - {k}: {v}")

# Rebuild model dari scratch dengan parameter eksplisit
# ← Fix: XGBoost 2.1.1 tidak kompatibel dengan ClassifierChain.predict_proba
#         solusinya rebuild dari scratch dengan objective eksplisit
base_xgb = XGBClassifier(
    objective='binary:logistic',  # ← eksplisit paksa classifier
    eval_metric='logloss',
    n_estimators    = best_params.get('n_estimators',     300),
    max_depth       = best_params.get('max_depth',        3),
    learning_rate   = best_params.get('learning_rate',    0.05),
    subsample       = best_params.get('subsample',        0.8),
    colsample_bytree= best_params.get('colsample_bytree', 0.7),
    gamma           = best_params.get('gamma',            0.2),
    min_child_weight= best_params.get('min_child_weight', 1),
    random_state    = 42,
    n_jobs          = -1
)

final_model = ClassifierChain(base_xgb, order='random', random_state=42)
print(f"✓ Model rebuilt: {type(final_model).__name__}")

# ==========================================
# 4. HELPER FUNCTIONS
# ==========================================
def chain_predict_proba(model, X):
    """
    Rekonstruksi manual predict_proba untuk ClassifierChain.
    Digunakan karena XGBoost 2.1.1 tidak kompatibel dengan
    ClassifierChain.predict_proba() dari sklearn.
    """
    X_arr      = X.values if hasattr(X, 'values') else X.copy()
    n_samples  = X_arr.shape[0]
    n_targets  = len(model.estimators_)
    all_probas = np.zeros((n_samples, n_targets))

    X_aug = X_arr.copy()
    for i, estimator in enumerate(model.estimators_):
        proba            = estimator.predict_proba(X_aug)[:, 1]
        all_probas[:, i] = proba
        pred_label       = (proba >= 0.5).astype(int).reshape(-1, 1)
        X_aug            = np.hstack([X_aug, pred_label])

    # Kembalikan format list of arrays seperti output predict_proba asli
    return [np.column_stack([1 - all_probas[:, i], all_probas[:, i]])
            for i in range(n_targets)]


def chain_predict_proba_threshold(model, X, thresholds):
    """
    Rekonstruksi predict_proba dengan threshold optimal.
    Augmentasi chain menggunakan threshold optimal (bukan 0.5).
    """
    X_arr      = X.values if hasattr(X, 'values') else X.copy()
    n_samples  = X_arr.shape[0]
    n_targets  = len(model.estimators_)
    all_probas = np.zeros((n_samples, n_targets))

    X_aug = X_arr.copy()
    for i, estimator in enumerate(model.estimators_):
        proba            = estimator.predict_proba(X_aug)[:, 1]
        all_probas[:, i] = proba
        # ← gunakan threshold optimal untuk augmentasi chain
        pred_label       = (proba >= thresholds[i]).astype(int).reshape(-1, 1)
        X_aug            = np.hstack([X_aug, pred_label])

    return [np.column_stack([1 - all_probas[:, i], all_probas[:, i]])
            for i in range(n_targets)]

print("✓ Helper functions defined")

with mlflow.start_run(run_name="XGBoost_ClassifierChain_Threshold_Optimization"):

    # ==========================================
    # 5. FIT MODEL (HANYA TRAIN_SUB)
    # ==========================================
    print("\n⏳ 5. Training model di Train_sub...")

    try:
        final_model.fit(X_train, Y_train)
        print("✓ Training di Train_sub selesai")

        # Verifikasi objective setelah fit
        print("\n   Verifikasi objective setelah fit:")
        for i, est in enumerate(final_model.estimators_):
            obj       = getattr(est, 'objective', 'tidak ada')
            n_classes = getattr(est, 'n_classes_', 'tidak ada')
            print(f"   Estimator {i}: objective={obj}, n_classes={n_classes}")

    except Exception as e:
        print(f"❌ Error saat training: {e}")
        raise

    # ==========================================
    # 6. THRESHOLD TUNING (VALIDATION SET)
    # ==========================================
    print("\n🔍 6. Mencari Threshold Optimal (VALIDATION SET)")

    try:
        print("   Menjalankan predict_proba di validation set...")
        y_val_proba = chain_predict_proba(final_model, X_val)  # ← pakai helper
        print(f"   ✅ predict_proba berhasil! {len(y_val_proba)} labels")
    except Exception as e:
        print(f"   ❌ Error: {e}")
        raise

    best_thresholds      = []
    validation_f1_scores = {}

    for i, target_name in enumerate(target_cols):
        y_true = Y_val.iloc[:, i]
        y_prob = y_val_proba[i][:, 1]

        thresholds = np.arange(0.1, 0.9, 0.01)
        f1_scores  = []

        for t in thresholds:
            y_pred_temp = (y_prob >= t).astype(int)
            f1_scores.append(f1_score(y_true, y_pred_temp, zero_division=0))

        best_idx = np.argmax(f1_scores)
        best_t   = thresholds[best_idx]
        best_f1  = f1_scores[best_idx]

        best_thresholds.append(best_t)
        validation_f1_scores[target_name] = best_f1

        print(f"   → {target_name.upper()}: Threshold = {best_t:.2f} | F1 Val = {best_f1:.4f}")

        mlflow.log_param(f"threshold_{target_name}", round(best_t, 2))
        mlflow.log_metric(f"validation_f1_{target_name}", best_f1)

    # ==========================================
    # 7. RETRAIN MODEL (FULL TRAIN)
    # ==========================================
    print("\n🔁 7. Retrain model di FULL TRAIN DATA...")

    try:
        final_model.fit(X_full_train, Y_full_train)
        print("✓ Retraining di full train selesai")
    except Exception as e:
        print(f"❌ Error saat retraining: {e}")
        raise

    # ==========================================
    # 8. EVALUASI FINAL DI TEST SET
    # ==========================================
    print("\n🏆 8. Evaluasi Final di TEST SET (UNSEEN DATA)")

    try:
        print("   Menjalankan predict_proba di test set...")
        y_test_proba = chain_predict_proba_threshold(
            final_model, X_test, best_thresholds  # ← pakai threshold optimal
        )
        print(f"   ✅ predict_proba berhasil! {len(y_test_proba)} labels")
    except Exception as e:
        print(f"❌ Error: {e}")
        raise

    y_test_optimized = np.zeros((len(X_test), len(target_cols)))
    for i in range(len(target_cols)):
        y_prob                 = y_test_proba[i][:, 1]
        y_test_optimized[:, i] = (y_prob >= best_thresholds[i]).astype(int)

    # ==========================================
    # 9. CALCULATE COMPREHENSIVE METRICS
    # ==========================================
    print("\n📊 9. Menghitung Metrik Evaluasi...")

    macro_f1    = f1_score(Y_test, y_test_optimized, average='macro',  zero_division=0)
    micro_f1    = f1_score(Y_test, y_test_optimized, average='micro',  zero_division=0)
    hamming     = hamming_loss(Y_test, y_test_optimized)
    exact_match = accuracy_score(Y_test, y_test_optimized)

    per_label_f1_scores = {}
    for i, target_name in enumerate(target_cols):
        f1 = f1_score(Y_test.iloc[:, i], y_test_optimized[:, i], zero_division=0)
        per_label_f1_scores[target_name] = f1

    print("-" * 60)
    print(f"🌟 FINAL MACRO F1-SCORE : {macro_f1:.4f}")
    print(f"🌟 FINAL MICRO F1-SCORE : {micro_f1:.4f}")
    print(f"   Hamming Loss         : {hamming:.4f}")
    print(f"   Exact Match Accuracy : {exact_match:.4f}")
    print("-" * 60)

    print("\nPer-Label F1 Scores:")
    for label, f1_val in per_label_f1_scores.items():
        print(f"   → {label}: {f1_val:.4f}")

    # ==========================================
    # 10. CLASSIFICATION REPORT
    # ==========================================
    print("\n📋 Classification Report:")
    class_report = classification_report(
        Y_test, y_test_optimized,
        target_names=['Depression', 'Anxiety', 'Stress'],
        zero_division=0
    )
    print(class_report)

    # ==========================================
    # 11. MLflow LOGGING
    # ==========================================
    print("\n📈 11. Logging ke MLflow...")

    # Parameters
    mlflow.log_param("strategy",              "Threshold Optimization with ClassifierChain")
    mlflow.log_param("val_split_ratio",       0.2)
    mlflow.log_param("num_features",          len(selected_features))
    mlflow.log_param("num_labels",            len(target_cols))
    mlflow.log_param("threshold_search_range","0.1-0.9")
    mlflow.log_param("threshold_search_step", 0.01)
    for k, v in best_params.items():
        mlflow.log_param(f"xgb_{k}", v)

    # Metrics
    mlflow.log_metric("test_macro_f1",             macro_f1)
    mlflow.log_metric("test_micro_f1",             micro_f1)
    mlflow.log_metric("test_hamming_loss",         hamming)
    mlflow.log_metric("test_exact_match_accuracy", exact_match)
    for label, f1_val in per_label_f1_scores.items():
        mlflow.log_metric(f"test_f1_{label}", f1_val)
    mlflow.log_metric("train_sub_samples",  X_train.shape[0])
    mlflow.log_metric("validation_samples", X_val.shape[0])
    mlflow.log_metric("test_samples",       X_test.shape[0])

    # Artifacts
    report_path = "classification_report.txt"
    with open(report_path, "w") as f:
        f.write(class_report)
    mlflow.log_artifact(report_path)

    thresholds_path = "optimal_thresholds.txt"
    with open(thresholds_path, "w") as f:
        f.write("Optimal Thresholds per Label:\n")
        f.write("-" * 40 + "\n")
        for label, threshold in zip(target_cols, best_thresholds):
            f.write(f"{label}: {threshold:.4f}\n")
    mlflow.log_artifact(thresholds_path)

    print("✓ MLflow logging complete")

    # ==========================================
    # 12. SUMMARY
    # ==========================================
    print("\n" + "="*70)
    print("🎉 THRESHOLD OPTIMIZATION SELESAI")
    print("="*70)
    print(f"Model: XGBoost + ClassifierChain")
    print(f"\nValidation Set F1-Scores:")
    for label, f1_val in validation_f1_scores.items():
        print(f"   {label}: {f1_val:.4f}")
    print(f"\nTest Set Results:")
    print(f"   Macro F1-Score : {macro_f1:.4f}")
    print(f"   Micro F1-Score : {micro_f1:.4f}")
    print(f"   Hamming Loss   : {hamming:.4f}")
    print(f"   Exact Match    : {exact_match:.4f}")
    print(f"\nOptimal Thresholds:")
    for label, threshold in zip(target_cols, best_thresholds):
        print(f"   {label}: {threshold:.4f}")
    print("="*70)

⏳ 1. Memuat Data...
✓ Full Train : (8724, 31)
✓ Test       : (5436, 31)

🔀 2. Split Train → Train_sub & Validation (80-20)
✓ Train_sub  : (6979, 31)
✓ Validation : (1745, 31)
✓ Test       : (5436, 31)

📦 3. Load Model & Rebuild dari Scratch...
✓ Best params dari pickle:
   - n_estimators: 100
   - max_depth: 3
   - learning_rate: 0.2
   - subsample: 0.7
   - colsample_bytree: 1.0
   - gamma: 0.5
   - min_child_weight: 5
✓ Model rebuilt: ClassifierChain
✓ Helper functions defined

⏳ 5. Training model di Train_sub...
✓ Training di Train_sub selesai

   Verifikasi objective setelah fit:
   Estimator 0: objective=binary:logistic, n_classes=2
   Estimator 1: objective=binary:logistic, n_classes=2
   Estimator 2: objective=binary:logistic, n_classes=2

🔍 6. Mencari Threshold Optimal (VALIDATION SET)
   Menjalankan predict_proba di validation set...
   ✅ predict_proba berhasil! 3 labels
   → RISK_DEPRESSION: Threshold = 0.35 | F1 Val = 0.7564
   → RISK_ANXIETY: Threshold = 0.18 | F1 Val = 0.7

In [17]:
# Cek distribusi validation vs test
print("Distribusi Validation:")
print(Y_val.mean().round(4))

print("\nDistribusi Test:")
print(Y_test.mean().round(4))

Distribusi Validation:
risk_depression    0.4126
risk_anxiety       0.4447
risk_stress        0.3490
dtype: float64

Distribusi Test:
risk_depression    0.7068
risk_anxiety       0.7349
risk_stress        0.6078
dtype: float64


In [18]:
# ==========================================
# 2. LOAD DATA ASLI UNTUK THRESHOLD SEARCH
# ==========================================

# Load data train ASLI (sebelum SMOTE)
train_ori = pd.read_csv(root_path / "Data/split/train_data.csv")

X_train_ori = train_ori[selected_features]
Y_train_ori = train_ori[target_cols].astype(int)

# Split dari data ASLI untuk validation
X_train_sub, X_val_ori, Y_train_sub, Y_val_ori = train_test_split(
    X_train_ori, Y_train_ori,
    test_size=0.2,
    random_state=42
)

print(f"✓ Train sub (asli) : {X_train_sub.shape}")
print(f"✓ Validation (asli): {X_val_ori.shape}")

# Cek distribusi validation vs test
print("\nDistribusi Validation (asli):")
print(Y_val_ori.mean().round(4))
print("\nDistribusi Test:")
print(Y_test.mean().round(4))

✓ Train sub (asli) : (17392, 31)
✓ Validation (asli): (4348, 31)

Distribusi Validation (asli):
risk_depression    0.7150
risk_anxiety       0.7364
risk_stress        0.6118
dtype: float64

Distribusi Test:
risk_depression    0.7068
risk_anxiety       0.7349
risk_stress        0.6078
dtype: float64


In [19]:
# Fit model di train SMOTE (tetap pakai SMOTE untuk training)
# Tapi cari threshold di validation ASLI
final_model.fit(X_full_train, Y_full_train)  # ← tetap pakai SMOTE

# Cari threshold di validation ASLI
y_val_proba = chain_predict_proba(final_model, X_val_ori)  # ← pakai data asli

best_thresholds      = []
validation_f1_scores = {}

for i, target_name in enumerate(target_cols):
    y_true = Y_val_ori.iloc[:, i]  # ← pakai label asli
    y_prob = y_val_proba[i][:, 1]

    thresholds = np.arange(0.1, 0.9, 0.01)
    f1_scores  = []

    for t in thresholds:
        y_pred_temp = (y_prob >= t).astype(int)
        f1_scores.append(f1_score(y_true, y_pred_temp, zero_division=0))

    best_idx = np.argmax(f1_scores)
    best_t   = thresholds[best_idx]
    best_f1  = f1_scores[best_idx]

    best_thresholds.append(best_t)
    validation_f1_scores[target_name] = best_f1

    print(f"   → {target_name.upper()}: Threshold = {best_t:.2f} | F1 Val = {best_f1:.4f}")

   → RISK_DEPRESSION: Threshold = 0.18 | F1 Val = 0.8660
   → RISK_ANXIETY: Threshold = 0.10 | F1 Val = 0.8692
   → RISK_STRESS: Threshold = 0.10 | F1 Val = 0.8080


In [20]:
# Buat model sementara khusus untuk cari threshold
# Dilatih di data ASLI bukan SMOTE
from xgboost import XGBClassifier
from sklearn.multioutput import ClassifierChain

print("🔧 Training model sementara di data ASLI untuk threshold search...")

base_xgb_ori = XGBClassifier(
    objective     = 'binary:logistic',
    eval_metric   = 'logloss',
    n_estimators  = best_params.get('n_estimators',     300),
    max_depth     = best_params.get('max_depth',        3),
    learning_rate = best_params.get('learning_rate',    0.05),
    subsample     = best_params.get('subsample',        0.8),
    colsample_bytree = best_params.get('colsample_bytree', 0.7),
    gamma         = best_params.get('gamma',            0.2),
    min_child_weight = best_params.get('min_child_weight', 1),
    random_state  = 42,
    n_jobs        = -1
)

# ← latih di data ASLI (train_sub asli, bukan SMOTE)
model_for_threshold = ClassifierChain(base_xgb_ori, order='random', random_state=42)
model_for_threshold.fit(X_train_sub, Y_train_sub)

print("✓ Model sementara selesai dilatih di data asli")

# Cari threshold di validation ASLI
print("\n🔍 Mencari threshold optimal di validation ASLI...")
y_val_proba = chain_predict_proba(model_for_threshold, X_val_ori)

best_thresholds      = []
validation_f1_scores = {}

for i, target_name in enumerate(target_cols):
    y_true = Y_val_ori.iloc[:, i]
    y_prob = y_val_proba[i][:, 1]

    thresholds = np.arange(0.1, 0.9, 0.01)
    f1_scores  = []

    for t in thresholds:
        y_pred_temp = (y_prob >= t).astype(int)
        f1_scores.append(f1_score(y_true, y_pred_temp, zero_division=0))

    best_idx = np.argmax(f1_scores)
    best_t   = thresholds[best_idx]
    best_f1  = f1_scores[best_idx]

    best_thresholds.append(best_t)
    validation_f1_scores[target_name] = best_f1

    print(f"   → {target_name.upper()}: Threshold = {best_t:.2f} | F1 Val = {best_f1:.4f}")

# Setelah dapat threshold, tetap pakai model SMOTE untuk prediksi final
print("\n✅ Threshold optimal ditemukan dari distribusi asli")
print(f"   Threshold: {dict(zip(target_cols, [round(t,2) for t in best_thresholds]))}")
print("\n⚠️  Prediksi final tetap menggunakan model SMOTE (final_model)")

🔧 Training model sementara di data ASLI untuk threshold search...
✓ Model sementara selesai dilatih di data asli

🔍 Mencari threshold optimal di validation ASLI...
   → RISK_DEPRESSION: Threshold = 0.46 | F1 Val = 0.8659
   → RISK_ANXIETY: Threshold = 0.32 | F1 Val = 0.8690
   → RISK_STRESS: Threshold = 0.50 | F1 Val = 0.8247

✅ Threshold optimal ditemukan dari distribusi asli
   Threshold: {'risk_depression': np.float64(0.46), 'risk_anxiety': np.float64(0.32), 'risk_stress': np.float64(0.5)}

⚠️  Prediksi final tetap menggunakan model SMOTE (final_model)
